In [ ]:
import pandas as pd


In [ ]:
web_df1 = pd.read_csv("../data/raw/df_final_web_data_pt_1.txt")
web_df1

In [ ]:
web_df2 = pd.read_csv("../data/raw/df_final_web_data_pt_2.txt")
web_df2

## Data Cleaning


As we can see have two dataframes with the same name columns and they need to be cleaned. The first thing we need to do, since they are, basically,
the same dataframe split in two files is to merge them. From there, we will start the cleaning.

In [ ]:
df_web = pd.concat([web_df1, web_df2], axis=0)
df_web = df_web.reset_index(drop=True)
df_web.shape


In [ ]:
#Check for nulls:
df_web.isnull().sum()

We are lucky! No nulls, but we can't celebrate yet, since even if it's filled, the data may be inconsistent


In [ ]:
df_web.info()

Client id should be an integer, correct, visitor and visit id should be strings as well as process step BUT date should be in date format

In [ ]:
df_web['date_time'] = pd.to_datetime(df_web['date_time'])
df_web.info()

In [ ]:
df_web.head()


### Now, we need to keep in mind that our analysis will be to understand the behaviour of the customer. For that we need to sort it chronologically

In [ ]:
df_web = df_web.sort_values(by=["client_id", "date_time"])
df_web.head(17)

In [ ]:
df_web.to_csv("df_final_web_data.csv", index=False)

## Analyse the client behaviour in the conversion rate from start to confirm:

In [ ]:
df_web.groupby("client_id")["process_step"].apply(list).head(20)

### There are some irregularities, like starting many times, going upwards and backwards, confirming two times...

In [ ]:
# Remove consecutive steps: 
df_web["prev_step"] = df_web.groupby("visit_id")["process_step"].shift()

df_web_clean = df_web[df_web["process_step"] != df_web["prev_step"]]


In [ ]:
funnel_df = df_web_clean.pivot_table(
    index=["visit_id", "client_id"],
    columns="process_step",
    values="date_time",
    aggfunc="min"
)

In [ ]:
funnel_df["converted"] = funnel_df["confirm"].notna()

### How many people reached the confirm step on average?:

In [ ]:
conversion_rate = funnel_df["converted"].mean()
conversion_rate

### 57% of the people, on average, reached the confirm step. These are the average on the diferent step process:

In [ ]:
funnel_df.notna().mean()

### More than 8% of the population didn't start on start, which is the "normal" order. That might be because of uncomplete data or that somehow the process could be started in another step via direct access. Though, we think it's important to mention it to help developers
